## Audit-Balance-Control (ABC) Checks — Bronze Layer

| Check | Scope | Rule | Failure Action |
|-------|-------|------|----------------|
| **AUDIT** | `card_transactions` | Record count per batch in **400 K – 600 K** range | Log + halt Silver |
| **BALANCE** | `neft_rtgs_transactions` | `SUM(amount)` per `msg_id` **= `_total_settlement_amt`** (± 0.01 tolerance) | Log + halt Silver |
| **CONTROL** | Both tables | **No duplicate** `txn_id` / `instr_id` within source; no cross-source collisions | Log + halt Silver |

> **Regulatory**: All results persisted to `bfsi.bronze_layer.abc_audit_log` with **7-year** retention.  
> **Orchestration**: On failure, sets `abc_check_passed = False` via `dbutils.jobs.taskValues` for Airflow `BranchOperator`.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, BooleanType, LongType, DoubleType
from datetime import datetime
import json

# ── Configuration ──────────────────────────────────────────────────────
CATALOG          = "bfsi"
SCHEMA           = "bronze_layer"
AUDIT_LOG_TABLE  = f"{CATALOG}.{SCHEMA}.abc_audit_log"

# Thresholds (test environment – production: 400_000 / 600_000)
CARD_MIN_RECORDS = 5_000
CARD_MAX_RECORDS = 50_000
BALANCE_TOLERANCE = 0.01          # ± tolerance for floating-point comparison

# Run metadata
RUN_TS   = datetime.utcnow()
RUN_ID   = RUN_TS.strftime("ABC_%Y%m%d_%H%M%S")

# Accumulator for all check results
abc_results = []

print(f"ABC Run: {RUN_ID}  |  Timestamp: {RUN_TS.isoformat()}")
print(f"Card record threshold: {CARD_MIN_RECORDS:,} – {CARD_MAX_RECORDS:,}  (test mode)")

In [0]:
# ── AUDIT CHECK: Card transaction record count per batch ──────────────
card_df = spark.table(f"{CATALOG}.{SCHEMA}.card_transactions")

audit_card = (
    card_df
    .groupBy("_batch_id", "_file_name", "_source_system")
    .agg(F.count("*").alias("record_count"))
    .withColumn(
        "audit_passed",
        F.col("record_count").between(CARD_MIN_RECORDS, CARD_MAX_RECORDS)
    )
)

audit_rows = audit_card.collect()

for row in audit_rows:
    passed = row["audit_passed"]
    detail = {
        "batch_id": row["_batch_id"],
        "file_name": row["_file_name"],
        "record_count": row["record_count"],
        "expected_range": f"{CARD_MIN_RECORDS}-{CARD_MAX_RECORDS}",
    }
    abc_results.append({
        "run_id": RUN_ID,
        "check_type": "AUDIT",
        "source_system": row["_source_system"],
        "check_name": "card_record_count",
        "passed": passed,
        "detail": json.dumps(detail),
        "check_ts": RUN_TS,
    })
    status = "PASS" if passed else "FAIL"
    print(f"AUDIT | {row['_batch_id']} | count={row['record_count']:,} | {status}")

In [0]:
# ── BALANCE CHECK: NEFT/RTGS settlement amount reconciliation ─────────
neft_df = spark.table(f"{CATALOG}.{SCHEMA}.neft_rtgs_transactions")

balance_check = (
    neft_df
    .groupBy("msg_id", "_file_name", "_batch_id", "_source_system", "_total_settlement_amt")
    .agg(
        F.sum("amount").alias("sum_txn_amounts"),
        F.count("*").alias("txn_count"),
    )
    .withColumn(
        "difference",
        F.abs(F.col("sum_txn_amounts") - F.col("_total_settlement_amt"))
    )
    .withColumn(
        "balance_passed",
        F.col("difference") <= BALANCE_TOLERANCE
    )
)

balance_rows = balance_check.collect()

for row in balance_rows:
    passed = row["balance_passed"]
    detail = {
        "msg_id": row["msg_id"],
        "batch_id": row["_batch_id"],
        "file_name": row["_file_name"],
        "sum_txn_amounts": row["sum_txn_amounts"],
        "header_settlement_amt": row["_total_settlement_amt"],
        "difference": row["difference"],
        "txn_count": row["txn_count"],
        "tolerance": BALANCE_TOLERANCE,
    }
    abc_results.append({
        "run_id": RUN_ID,
        "check_type": "BALANCE",
        "source_system": row["_source_system"],
        "check_name": "neft_rtgs_settlement_reconciliation",
        "passed": passed,
        "detail": json.dumps(detail),
        "check_ts": RUN_TS,
    })
    status = "PASS" if passed else "FAIL"
    print(
        f"BALANCE | {row['msg_id']} | SUM={row['sum_txn_amounts']:,.2f} "
        f"vs HDR={row['_total_settlement_amt']:,.2f} | diff={row['difference']:.4f} | {status}"
    )

In [0]:
# ── CONTROL CHECK: Duplicate detection ────────────────────────────────

# 1. Intra-source duplicates: card_transactions
card_dupes = (
    card_df
    .groupBy("txn_id", "_source_system")
    .agg(F.count("*").alias("cnt"))
    .filter(F.col("cnt") > 1)
)
card_dupe_count = card_dupes.count()

# 2. Intra-source duplicates: neft_rtgs_transactions
neft_dupes = (
    neft_df
    .groupBy("instr_id", "_source_system")
    .agg(F.count("*").alias("cnt"))
    .filter(F.col("cnt") > 1)
)
neft_dupe_count = neft_dupes.count()

# 3. Cross-source duplicates (txn_id vs instr_id collision)
cross_dupes = (
    card_df.select(F.col("txn_id").alias("id"), F.lit("CARD").alias("src"))
    .join(
        neft_df.select(F.col("instr_id").alias("id"), F.lit("NEFT_RTGS").alias("src")),
        on="id",
        how="inner",
    )
)
cross_dupe_count = cross_dupes.count()

for label, dupe_cnt, source in [
    ("card_intra_source_duplicates", card_dupe_count, "CARD"),
    ("neft_rtgs_intra_source_duplicates", neft_dupe_count, "NEFT_RTGS"),
    ("cross_source_id_collision", cross_dupe_count, "ALL"),
]:
    passed = dupe_cnt == 0
    detail = {"duplicate_id_count": dupe_cnt}
    # Capture sample duplicate IDs for debugging (max 10)
    if not passed:
        if "card_intra" in label:
            samples = [r["txn_id"] for r in card_dupes.limit(10).collect()]
        elif "neft_rtgs_intra" in label:
            samples = [r["instr_id"] for r in neft_dupes.limit(10).collect()]
        else:
            samples = [r["id"] for r in cross_dupes.limit(10).collect()]
        detail["sample_ids"] = samples

    abc_results.append({
        "run_id": RUN_ID,
        "check_type": "CONTROL",
        "source_system": source,
        "check_name": label,
        "passed": passed,
        "detail": json.dumps(detail),
        "check_ts": RUN_TS,
    })
    status = "PASS" if passed else "FAIL"
    print(f"CONTROL | {label} | duplicates={dupe_cnt:,} | {status}")

In [0]:
# ── Persist results to abc_audit_log (7-year regulatory retention) ─────
results_schema = StructType([
    StructField("run_id", StringType()),
    StructField("check_type", StringType()),
    StructField("source_system", StringType()),
    StructField("check_name", StringType()),
    StructField("passed", BooleanType()),
    StructField("detail", StringType()),
    StructField("check_ts", TimestampType()),
])

results_df = spark.createDataFrame(abc_results, schema=results_schema)

# Create table with 7-year retention if it doesn't exist
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {AUDIT_LOG_TABLE} (
        run_id          STRING,
        check_type      STRING,
        source_system   STRING,
        check_name      STRING,
        passed          BOOLEAN,
        detail          STRING,
        check_ts        TIMESTAMP
    )
    USING DELTA
    COMMENT 'ABC check audit log – regulatory 7-year retention'
    TBLPROPERTIES (
        'delta.deletedFileRetentionDuration' = 'interval 2555 days',
        'delta.logRetentionDuration'         = 'interval 2555 days'
    )
""")

results_df.write.mode("append").saveAsTable(AUDIT_LOG_TABLE)

print(f"\n Logged {len(abc_results)} check result(s) to {AUDIT_LOG_TABLE}")
display(results_df)

In [0]:
# ── Evaluate overall status & signal Airflow BranchOperator ────────────
all_passed = all(r["passed"] for r in abc_results)
failed_checks = [r for r in abc_results if not r["passed"]]

print("=" * 70)
if all_passed:
    print("ALL ABC CHECKS PASSED — Silver job may proceed.")
else:
    print("ABC CHECK FAILURE(S) DETECTED — Silver job will be halted.")
    for f in failed_checks:
        print(f"    ▸ [{f['check_type']}] {f['check_name']} | {f['source_system']}")
        print(f"      Detail: {f['detail']}")
print("=" * 70)

# Set task value for Airflow BranchOperator integration
# Airflow reads this via `databricks.sdk` or the Jobs API to decide branching
try:
    dbutils.jobs.taskValues.set(key="abc_check_passed", value=all_passed)
    dbutils.jobs.taskValues.set(key="abc_run_id",       value=RUN_ID)
    print(f"\n🔗 Task values set: abc_check_passed={all_passed}, abc_run_id={RUN_ID}")
except Exception:
    # Running interactively (not as a job task)
    print(f"\n Interactive mode — task values not set (abc_check_passed={all_passed})")

# Fail the notebook if any check failed (stops downstream Airflow tasks)
if not all_passed:
    raise Exception(
        f"ABC Check Failed — {len(failed_checks)} failure(s) detected. "
        f"Run ID: {RUN_ID}. See {AUDIT_LOG_TABLE} for details."
    )